# 08 — Espectrogramas de mel: antes y después del preprocesado

Genera las figuras de espectrogramas para la sección 4.1 de la memoria (efecto del preprocesado sobre las representaciones espectrales).

- **Mic fijo**: `data/audios/` (raw) vs `data/clean/` (Wiener + HPSS).
- **Móvil**: audio crudo vs audio tras Wiener gated (hp=100 Hz, nr=0.50, ×2).

> **Kernel requerido:** `.venv311` (Python 3.11). Librosa y miniaudio deben estar instalados.
> `Ctrl+Shift+P` → `Python: Select Interpreter` → seleccionar `.venv311`.

In [ ]:
import sys, warnings, tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa as lb
import librosa.display
import soundfile as sf

warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

SR      = 16_000
N_FFT   = 2048
HOP     = 256
N_MELS  = 128
FMIN    = 0
FMAX    = 8000
OUT_DIR = ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def wave_to_mel_db(audio):
    X   = np.abs(lb.stft(audio, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT))
    mel = lb.feature.melspectrogram(sr=SR, S=X, n_fft=N_FFT, hop_length=HOP,
                                    power=1.0, n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
                                    htk=True, norm=None)
    return lb.amplitude_to_db(mel, ref=np.max)

print(f'Python: {sys.version[:6]}  |  librosa: {lb.__version__}')

## Parte A — Micrófono fijo (Wiener + HPSS)

Selección automática de un archivo representativo (RMS > 0.005) de `data/audios/` y su par preprocesado en `data/clean/`.

In [ ]:
RAW_DIR   = ROOT / 'data' / 'audios'
CLEAN_DIR = ROOT / 'data' / 'clean'

np.random.seed(42)
raw_files   = sorted(RAW_DIR.glob('*.wav'))
clean_files = {f.name for f in CLEAN_DIR.glob('*.wav')}
matched     = [f for f in raw_files if f.name in clean_files]
print(f'Archivos pareados raw+clean: {len(matched)}')

chosen_raw = chosen_clean = None
for f in np.random.choice(matched, size=min(50, len(matched)), replace=False):
    audio, _ = lb.load(str(f), sr=SR, mono=True)
    if np.sqrt(np.mean(audio**2)) > 0.005:
        chosen_raw   = Path(f)
        chosen_clean = CLEAN_DIR / Path(f).name
        print(f'Seleccionado: {Path(f).name}  RMS={np.sqrt(np.mean(audio**2)):.4f}')
        break

if chosen_raw is None:
    chosen_raw   = matched[0]
    chosen_clean = CLEAN_DIR / matched[0].name
    print(f'Fallback: {matched[0].name}')

In [ ]:
audio_raw,   _ = lb.load(str(chosen_raw),   sr=SR, mono=True)
audio_clean, _ = lb.load(str(chosen_clean), sr=SR, mono=True)

mel_raw   = wave_to_mel_db(audio_raw)
mel_clean = wave_to_mel_db(audio_clean)

vmin_shared = max(mel_raw.min(), mel_clean.min())
t_axis = np.linspace(0, len(audio_raw) / SR, len(audio_raw))

def plot_spectrogram_single(audio, mel, title, color, fname, vmin=-80.0):
    fig = plt.figure(figsize=(12, 6))
    gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.4)
    ax_w = fig.add_subplot(gs[0])
    ax_m = fig.add_subplot(gs[1])

    t = np.linspace(0, len(audio) / SR, len(audio))
    ax_w.plot(t, audio, lw=0.5, color=color, alpha=0.85)
    ax_w.set_title(title, fontsize=12)
    ax_w.set_ylabel('Amplitud')
    ax_w.set_xlabel('Tiempo (s)')
    ax_w.set_xlim(0, len(audio) / SR)
    ax_w.text(0.02, 0.94, f'RMS = {np.sqrt(np.mean(audio**2)):.4f}',
              transform=ax_w.transAxes, fontsize=9, color='gray', va='top')

    im = lb.display.specshow(mel, sr=SR, hop_length=HOP, x_axis='time', y_axis='mel',
                              fmax=FMAX, ax=ax_m, cmap='magma', vmin=vmin, vmax=0.0)
    ax_m.set_title('Espectrograma de mel', fontsize=12)
    ax_m.set_ylabel('Frecuencia (Hz)')
    fig.colorbar(im, ax=ax_m, orientation='vertical', label='Amplitud relativa (dB)',
                 shrink=0.9, pad=0.02)

    plt.savefig(OUT_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: outputs/{fname}')

# Figura A: Audio original del prototipo
plot_spectrogram_single(audio_raw,   mel_raw,
    f'Audio original — prototipo (mic fijo) | {chosen_raw.name}',
    '#2c3e50', 'spectrogram_mic_before.png', vmin=vmin_shared)

# Figura B: Audio preprocesado del prototipo
plot_spectrogram_single(audio_clean, mel_clean,
    f'Audio preprocesado — prototipo (Declip + Wiener + HPSS) | {chosen_raw.name}',
    '#27ae60', 'spectrogram_mic_after.png', vmin=vmin_shared)

## Parte B — Móvil (Wiener gated)

Extrae un fragmento del audio MP3/MP4 del móvil y aplica el preprocesado Wiener gated (hp=100 Hz, nr=0.50, passes=2).
Requiere `miniaudio` instalado en `.venv311`.

In [ ]:
try:
    import miniaudio
    HAS_MINIAUDIO = True
except ImportError:
    HAS_MINIAUDIO = False
    print('[WARN] miniaudio no instalado — pip install miniaudio')

MOBILE_DIR = ROOT / 'data' / 'mobile'
AUDIO_EXTS = ['.mp3', '.mp4', '.m4a', '.wav']

def find_audio(d):
    for ext in AUDIO_EXTS:
        m = list(d.glob(f'*{ext}'))
        if m: return m[0]
    return None

sessions = [(d, find_audio(d)) for d in MOBILE_DIR.iterdir() if d.is_dir()]
sessions = [(d, a) for d, a in sessions if a is not None]
print(f'Sesiones con audio: {[d.name for d, a in sessions]}')

TARGET = 'SILLA-PAIPORTA_1'
match  = [(d, a) for d, a in sessions if d.name == TARGET]
session_dir, audio_path = match[0] if match else sessions[0]
print(f'Sesión seleccionada: {session_dir.name}  |  {audio_path.name}')

In [ ]:
if not HAS_MINIAUDIO:
    print('[SKIP] necesita miniaudio')
else:
    from clean_audio import clean_audio as wiener_clean

    CHUNK_SEC  = 10
    CHUNK_SAMP = SR * CHUNK_SEC
    T_START    = 200   # cambiar para otro fragmento representativo
    WIENER     = dict(hp_cutoff=100, nr_strength=0.50, lp_cutoff=8000, passes=2)

    print('Cargando audio móvil...')
    decoded = miniaudio.decode_file(str(audio_path),
                output_format=miniaudio.SampleFormat.FLOAT32,
                nchannels=1, sample_rate=SR)
    audio_full = np.frombuffer(decoded.samples, dtype=np.float32).copy()
    print(f'Duración: {len(audio_full)/SR:.1f}s  |  fragmento t=[{T_START}s, {T_START+CHUNK_SEC}s]')

    audio_mob_raw = audio_full[T_START*SR : T_START*SR + CHUNK_SAMP].copy()

    with tempfile.TemporaryDirectory() as tmp:
        in_p  = str(Path(tmp) / 'in.wav')
        out_p = str(Path(tmp) / 'out.wav')
        sf.write(in_p, audio_mob_raw, SR, subtype='PCM_16')
        wiener_clean(in_p, out_p, **WIENER)
        audio_mob_clean, _ = lb.load(out_p, sr=SR, mono=True)

    mel_mob_raw   = wave_to_mel_db(audio_mob_raw)
    mel_mob_clean = wave_to_mel_db(audio_mob_clean)
    vmin_mob = max(mel_mob_raw.min(), mel_mob_clean.min())

    print(f'RMS raw={np.sqrt(np.mean(audio_mob_raw**2)):.4f}  '
          f'clean={np.sqrt(np.mean(audio_mob_clean**2)):.4f}  OK.')

In [ ]:
if not HAS_MINIAUDIO:
    print('[SKIP]')
else:
    # Figura C: Audio original del móvil
    plot_spectrogram_single(audio_mob_raw, mel_mob_raw,
        f'Audio original — móvil | {session_dir.name}  t=[{T_START}s, {T_START+CHUNK_SEC}s]',
        '#2c3e50', 'spectrogram_mobile_before.png', vmin=vmin_mob)

    # Figura D: Audio preprocesado del móvil
    plot_spectrogram_single(audio_mob_clean, mel_mob_clean,
        f'Audio preprocesado — móvil (Wiener gated, hp=100Hz, nr=0.50, ×2) | {session_dir.name}',
        '#e67e22', 'spectrogram_mobile_after.png', vmin=vmin_mob)